# Anarkali: check labels with an open LLM (GPU)

Qwen3 (Apache 2.0, commercial use allowed) reads every `page_type` and `page_status` row **without**
seeing the label and picks an option. A row is kept only when Qwen agrees with the label
(rule below). Other decisions are copied unchanged. The checked dataset goes to
`RUN_NAME/web-expert-v1-checked/` on HF, with `qwen-check-report.json` and a sample of dropped rows.

Kaggle: Accelerator GPU (T4 x2 is best), Internet on, secret HF_TOKEN.


In [ ]:
RUN_NAME = "run3"
HF_REPO = "toufiqqureshi651/anarkali-web"
SRC = "run3/web-expert-v1"
OUT_NAME = "web-expert-v1-checked"
MODEL = "Qwen/Qwen3-4B-Instruct-2507"   # Apache 2.0
CHECK = ("page_type", "page_status")
MAX_CHARS = 3500        # page outline characters shown to the LLM
TOKEN_BUDGET = 12000    # tokens per batch per GPU
# Keep rule: a label from one source ("single") must be Qwen's first choice;
# a label two sources agreed on ("agreed") must be in Qwen's top 2.

In [ ]:
import os, json, subprocess, sys, time, hashlib, random
from collections import Counter, defaultdict
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
DATA, OUT = WORK / "src", WORK / OUT_NAME
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

def _hf_token():
    token = os.environ.get("HF_TOKEN")
    for getter in ("kaggle", "colab"):
        if token:
            break
        try:
            if getter == "kaggle":
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("HF_TOKEN")
            else:
                from google.colab import userdata
                token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face token (hf_...): ").strip()
    return token

from huggingface_hub import HfApi, hf_hub_download
HF_TOKEN = _hf_token()
hf = HfApi(token=HF_TOKEN)
SPLITS = ["train", "development", "calibration", "test"]
for name in SPLITS + ["manifest"]:
    f = f"{name}.json" if name == "manifest" else f"{name}.jsonl"
    p = hf_hub_download(HF_REPO, f"{SRC}/{f}", token=HF_TOKEN)
    (DATA / f).write_bytes(Path(p).read_bytes())
import torch
GPUS = torch.cuda.device_count()
print("data ready | GPUs:", GPUS, [torch.cuda.get_device_name(i) for i in range(GPUS)])
assert GPUS, "turn on a GPU accelerator"

In [ ]:
WORKER_SRC = '\n"""One GPU: score each job\'s options with one forward pass (no generation) and write results."""\nimport json, sys, torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nmodel_id, jobs_path, out_path, gpu, max_chars, token_budget = sys.argv[1:7]\ngpu, max_chars, token_budget = int(gpu), int(max_chars), int(token_budget)\ntok = AutoTokenizer.from_pretrained(model_id)\ntok.padding_side = "left"\nmodel = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, attn_implementation="sdpa").to(f"cuda:{gpu}").eval()\nLETTERS = [chr(ord("A") + i) for i in range(26)]\nletter_ids = [tok.encode(l, add_special_tokens=False) for l in LETTERS]\nassert all(len(i) == 1 for i in letter_ids), "letters must be single tokens"\nletter_ids = [i[0] for i in letter_ids]\n\ndef prompt(job):\n    s = job["state"]\n    opts = "\\n".join(f"{LETTERS[i]}. {c[\'text\']}" for i, c in enumerate(job["candidates"]))\n    text = (f"You are labelling web pages. Read the page and answer the question.\\n\\n"\n            f"URL: {s.get(\'url\', \'\')}\\nTitle: {s.get(\'title\', \'\')}\\n"\n            f"Page outline (tags show structure; <nav>, <footer>, <form> lines are summarised):\\n"\n            f"{str(s.get(\'page\', \'\'))[:max_chars]}\\n\\n"\n            f"Question: {job[\'question\']}\\nOptions:\\n{opts}\\n\\nAnswer with the letter of the best option only.")\n    return tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)\n\njobs = [json.loads(l) for l in open(jobs_path, encoding="utf-8")]\nenc = [(j, tok(prompt(j), add_special_tokens=False)["input_ids"]) for j in jobs]\nenc.sort(key=lambda x: len(x[1]))\ndone = 0\nwith open(out_path, "w", encoding="utf-8") as out, torch.inference_mode():\n    i = 0\n    while i < len(enc):\n        batch = enc[i:i + max(1, token_budget // len(enc[i][1]))]   # sorted by length: trim to fit\n        while len(batch) > 1 and len(batch) * len(batch[-1][1]) > token_budget:\n            batch = batch[:-1]\n        width = len(batch[-1][1])\n        ids = torch.full((len(batch), width), tok.pad_token_id, dtype=torch.long)\n        mask = torch.zeros_like(ids)\n        for b, (_, seq) in enumerate(batch):\n            ids[b, width - len(seq):] = torch.tensor(seq)\n            mask[b, width - len(seq):] = 1\n        logits = model(input_ids=ids.to(model.device), attention_mask=mask.to(model.device)).logits[:, -1].float()\n        for b, (job, _) in enumerate(batch):\n            k = len(job["candidates"])\n            probs = torch.softmax(logits[b, letter_ids[:k]], -1).tolist()\n            out.write(json.dumps({"case_id": job["case_id"], "probs": probs}) + "\\n")\n        out.flush()\n        i += len(batch)\n        done += len(batch)\n        if done // 500 != (done - len(batch)) // 500:\n            print(f"gpu{gpu}: {done}/{len(enc)}", flush=True)\nprint(f"gpu{gpu}: finished {done}", flush=True)\n'

def decision(r): return r["case_id"].rsplit("::", 1)[-1]
def gold(r): return max(range(len(r["target"])), key=r["target"].__getitem__)

rows = {s: [json.loads(l) for l in open(DATA / f"{s}.jsonl", encoding="utf-8")] for s in SPLITS}
jobs = []
for s in SPLITS:
    for r in rows[s]:
        if decision(r) in CHECK:
            state = r["state"] if isinstance(r["state"], dict) else json.loads(r["state"])
            jobs.append({"case_id": r["case_id"], "state": state, "question": r["question"],
                         "candidates": r["candidates"]})
print("rows to check:", len(jobs), Counter(j["case_id"].rsplit("::", 1)[-1] for j in jobs))

(WORK / "worker.py").write_text(WORKER_SRC)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.51", "accelerate"], check=True)
random.Random(0).shuffle(jobs)
procs = []
for g in range(GPUS):
    part = WORK / f"jobs{g}.jsonl"
    with open(part, "w", encoding="utf-8") as f:
        for j in jobs[g::GPUS]:
            f.write(json.dumps(j, ensure_ascii=False) + "\n")
    procs.append(subprocess.Popen([sys.executable, str(WORK / "worker.py"), MODEL, str(part), str(WORK / f"res{g}.jsonl"),
                                   str(g), str(MAX_CHARS), str(TOKEN_BUDGET)]))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(60)
    done = sum(sum(1 for _ in open(WORK / f"res{g}.jsonl")) for g in range(GPUS) if (WORK / f"res{g}.jsonl").exists())
    print(f"{done}/{len(jobs)} checked, {(time.time() - t0) / 60:.0f} min", flush=True)
assert all(p.returncode == 0 for p in procs), [p.returncode for p in procs]

In [ ]:
probs = {}
for g in range(GPUS):
    for l in open(WORK / f"res{g}.jsonl", encoding="utf-8"):
        x = json.loads(l); probs[x["case_id"]] = x["probs"]
assert len(probs) == len(jobs), (len(probs), len(jobs))

stats = defaultdict(lambda: Counter())
dropped_sample, manifest = [], json.loads((DATA / "manifest.json").read_text())
for s in SPLITS:
    kept = []
    for r in rows[s]:
        if r["case_id"] not in probs:
            kept.append(r); continue
        p, g = probs[r["case_id"]], gold(r)
        order = sorted(range(len(p)), key=lambda i: -p[i])
        ok = order[0] == g if r.get("label_quality") != "agreed" else g in order[:2]
        key = f"{decision(r)}={r['candidates'][g]['id']}|{r.get('label_quality')}"
        stats[key]["kept" if ok else "dropped"] += 1
        if ok:
            kept.append(r)
        elif len(dropped_sample) < 400:
            st = r["state"] if isinstance(r["state"], dict) else json.loads(r["state"])
            dropped_sample.append({"split": s, "url": st.get("url"), "title": st.get("title"),
                                   "label": r["candidates"][g]["id"], "quality": r.get("label_quality"),
                                   "qwen": r["candidates"][order[0]]["id"], "qwen_p": round(p[order[0]], 3)})
    text = "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in kept)
    (OUT / f"{s}.jsonl").write_text(text, encoding="utf-8")
    c = manifest["split_counts"][s]
    c["decision_cases"] = len(kept)
    c["source_groups"] = len({r["source_group"] for r in kept})
    c["labels"] = dict(sorted(Counter(f"{decision(r)}={r['candidates'][gold(r)]['id']}" for r in kept).items()))
    c["label_quality"] = dict(Counter(r.get("label_quality") for r in kept))
    c["sha256"] = hashlib.sha256(text.encode("utf-8")).hexdigest()

report = {k: dict(v) | {"kept_fraction": round(v["kept"] / (v["kept"] + v["dropped"]), 3)} for k, v in sorted(stats.items())}
manifest["dataset"] = OUT_NAME
manifest["label_check"] = {"model": MODEL, "license": "Apache-2.0", "decisions": list(CHECK),
                           "rule": "single-source labels must be the LLM's first choice; agreed labels must be in its top 2",
                           "llm_input": f"url, title and the first {MAX_CHARS} characters of the page outline (no label)"}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
(WORK / "qwen-check-report.json").write_text(json.dumps({"by_label": report, "dropped_sample": dropped_sample}, indent=1, ensure_ascii=False), encoding="utf-8")
hf.upload_folder(repo_id=HF_REPO, folder_path=str(OUT), path_in_repo=f"{RUN_NAME}/{OUT_NAME}")
hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(WORK / "qwen-check-report.json"), path_in_repo=f"{RUN_NAME}/qwen-check-report.json")
tot = Counter()
for v in stats.values(): tot.update(v)
print("kept", tot["kept"], "dropped", tot["dropped"])
for k, v in report.items(): print(f"{k:45s} {v}")
for s in SPLITS: print(s, manifest["split_counts"][s]["decision_cases"])